## Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend

Task 4.1 — One validated call

In [1]:
!pip install -q google-genai pydantic

In [1]:
import json, time
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field
from google.genai.errors import APIError
from pydantic import ValidationError


In [2]:

API_KEY = userdata.get('gemini_key')
client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.5-flash"

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]
}

SYSTEM_TEXT = (
    "You are an expert healthcare operations analyst classifying patient call-centre "
    "no-show follow-up notes into strict, structured JSON categories."
)

def analyse_note(note_text: str, max_retries=3) -> dict:
    prompt = f"""
    Context: A medical clinic's front desk calls patients who missed their scheduled appointments to log the reason why.
    Role: Healthcare Operations Analyst.
    Action: Analyze the call-centre note below and classify the primary reason for the missed appointment into EXACTLY ONE of these allowed root_cause strings:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    Also provide:
    - confidence: A float between 0.0 and 1.0 indicating how clearly the note supports your chosen root_cause.
    - summary: A concise summary of the note in at most 25 words.
    - recommended_action: A practical, concrete next step for the clinic front desk.

    Tone: Objective, concise, and operational.
    Format: Return strictly valid JSON matching the requested schema.

    Call-centre note: \"{note_text}\"
    """
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                    thinking_config=types.ThinkingConfig(thinking_level="minimal")
                )
            )
            return json.loads(response.text)

        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) * 5  # 5s, then 10s, then 20s
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                raise e

notes_df = pd.read_csv("data/followup_notes.csv")
n001_row = notes_df[notes_df["note_id"] == "N001"].iloc[0]
n001_text = n001_row["note_text"]

print(f"Input Note (N001): {n001_text}\n")
result_n001 = analyse_note(n001_text)

print("Parsed JSON Output:")
print(json.dumps(result_n001, indent=2))

Input Note (N001): Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

Parsed JSON Output:
{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.98,
  "summary": "Patient forgot the appointment because she received no reminder message and requested to reschedule for next week.",
  "recommended_action": "Reschedule the patient's appointment for next week and verify contact details to ensure automated reminders are enabled."
}


Task 4.2 — Validate and process the 10 notes

In [3]:
ALLOWED_ROOT_CAUSES = {
    "forgot_or_no_reminder", "distance_or_transport",
    "cost_or_fee", "illness_or_emergency",
    "work_or_schedule", "other"
}

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

notes_df = pd.read_csv("data/followup_notes.csv")
results = []

print("Starting batch processing of notes...")

for idx, row in notes_df.head(10).iterrows():
    note_id = row.get("note_id", f"N00{idx+1}")
    appointment_id = row.get("appointment_id", "")
    note_text = row.get("note_text", None)

    print(f"Processing {note_id}...", end=" ")

    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": note_id, "appointment_id": appointment_id,
            "root_cause": "other", "confidence": 0.0,
            "summary": "No note recorded", "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank", "raw_response": ""
        })
        print("Skipped (blank note).")
        continue

    raw_resp_text, validation_status = "", "invalid"
    rc, conf, summ, rec = "", 0.0, "", ""

    try:
        response_data = analyse_note(note_text)

        rc = response_data.get("root_cause", "")
        conf = response_data.get("confidence", 0.0)
        summ = response_data.get("summary", "")
        rec = response_data.get("recommended_action", "")

        insight = FollowupInsight(**response_data)
        if insight.root_cause in ALLOWED_ROOT_CAUSES:
            validation_status = "valid"

        print(f"Success [{validation_status}]")

    except Exception as e:
        print(f"Error: {e}")

    results.append({
        "note_id": note_id, "appointment_id": appointment_id,
        "root_cause": rc, "confidence": conf, "summary": summ,
        "recommended_action": rec, "validation_status": validation_status,
        "raw_response": raw_resp_text
    })

    if idx < 9:
        time.sleep(13)

pd.DataFrame(results).to_csv("notes_analysed.csv", index=False)
print("\nDone! Saved to notes_analysed.csv")

Starting batch processing of notes...
Processing N001... Success [valid]
Processing N002... Success [valid]
Processing N003... Success [valid]
Processing N004... Skipped (blank note).
Processing N005... Success [valid]
Processing N006... Success [valid]
Processing N007... Success [valid]
Processing N008... Success [valid]
Processing N009... Success [valid]
Processing N010... Success [valid]

Done! Saved to notes_analysed.csv


Task 4.3 — Break it on purpose

In [ ]:
# (a) — Pydantic Validation Error Testing

In [4]:
# Test 1: Confidence out of bounds (1.7 > 1.0)
bad_data_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot appointment date.",
    "recommended_action": "Send SMS reminder 24 hours prior."
}

try:
    FollowupInsight(**bad_data_1)
except ValidationError as e:
    print("--- Validation Error Test 1 (Confidence Out of Range) ---")
    print(e)

print("\n" + "="*50 + "\n")

# Test 2: Missing required field ('recommended_action')
bad_data_2 = {
    "root_cause": "work_or_schedule",
    "confidence": 0.9,
    "summary": "Patient stuck at work shift."
}

try:
    FollowupInsight(**bad_data_2)
except ValidationError as e:
    print("--- Validation Error Test 2 (Missing Field) ---")
    print(e)

--- Validation Error Test 1 (Confidence Out of Range) ---
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


--- Validation Error Test 2 (Missing Field) ---
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'work_or_s...t stuck at work shift.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
# (b) — Prompt Comparison (V1 vs V2) on N007 & N009

In [5]:
# V1 Prompt (Minimal / Unconstrained)
def analyse_note_v1(note_text):
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=f"Classify this note: {note_text}",
        config=types.GenerateContentConfig(
            temperature=0, response_mime_type="application/json", response_schema=SCHEMA
        )
    )
    return json.loads(response.text)

# Run comparison for N007 and N009
comparison_results = []

for note_id, note_text in [("N007", notes_df.loc[6, "note_text"]), ("N009", notes_df.loc[8, "note_text"])]:
    # V1 test
    res_v1 = analyse_note_v1(note_text)
    rc_v1 = res_v1.get("root_cause", "")
    valid_v1 = "Valid" if rc_v1 in ALLOWED_ROOT_CAUSES else "Invalid"

    comparison_results.append({
        "Note ID": note_id,
        "Prompt Version": "V1 (Simple)",
        "Returned root_cause": rc_v1,
        "Allowed List Status": valid_v1
    })

    # V2 test
    res_v2 = analyse_note(note_text)
    rc_v2 = res_v2.get("root_cause", "")
    valid_v2 = "Valid" if rc_v2 in ALLOWED_ROOT_CAUSES else "Invalid"

    comparison_results.append({
        "Note ID": note_id,
        "Prompt Version": "V2 (CRAFT + Allowed List)",
        "Returned root_cause": rc_v2,
        "Allowed List Status": valid_v2
    })

comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_markdown(index=False))

| Note ID   | Prompt Version            | Returned root_cause                     | Allowed List Status   |
|:----------|:--------------------------|:----------------------------------------|:----------------------|
| N007      | V1 (Simple)               | SMS Delivery Failure                    | Invalid               |
| N007      | V2 (CRAFT + Allowed List) | forgot_or_no_reminder                   | Valid                 |
| N009      | V1 (Simple)               | Patient phone powered off / unreachable | Invalid               |
| N009      | V2 (CRAFT + Allowed List) | other                                   | Valid                 |


Task 4.4 — Recommendation and GitHub

In [6]:

notes_analysed = pd.read_csv("notes_analysed.csv")
appointments_clean = pd.read_csv("appointments_clean.csv")

merged_df = pd.merge(notes_analysed, appointments_clean, on="appointment_id", how="inner")

print(f"Merged table shape: {merged_df.shape}")
assert merged_df.shape[0] == 10, "Merged table must have exactly 10 rows."

crosstab_result = pd.crosstab(merged_df["root_cause"], merged_df["reminder_sent"])
print("\n--- Cross-Tabulation: Root Cause vs Reminder Sent ---")
print(crosstab_result)

Merged table shape: (10, 23)

--- Cross-Tabulation: Root Cause vs Reminder Sent ---
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1
